In [ ]:
import pandas as pd
import networkx as nx
import json
import numpy as np
from geopy.distance import distance as geopy_distance

random.seed(42)
np.random.seed(42)

# Load data
airports = pd.read_csv("airports.dat", header=None)
routes = pd.read_csv("routes.dat", header=None)
airlines = pd.read_csv("airlines.dat", header=None)

# Set column names
airports.columns = ['Airport ID', 'Name', 'City', 'Country', 'IATA', 'ICAO',
                    'Latitude', 'Longitude', 'Altitude', 'Timezone',
                    'DST', 'Tz database time zone', 'Type', 'Source']
routes.columns = ['Airline', 'Airline ID', 'Source Airport', 'Source Airport ID',
                  'Destination Airport', 'Destination Airport ID', 'Codeshare',
                  'Stops', 'Equipment']
airlines.columns = ['Airline ID', 'Name', 'Alias', 'IATA', 'ICAO', 'Callsign', 'Country', 'Active']

# Convert 'Airline ID' to string to match in both DataFrames
routes['Airline ID'] = routes['Airline ID'].astype(str)
airlines['Airline ID'] = airlines['Airline ID'].astype(str)

# Merge airline country into routes
routes = routes.merge(airlines[['Airline ID', 'Country']], on='Airline ID', how='left')

# Filter airlines from selected countries
allowed_countries = ['United States', 'Canada', 'United Kingdom', 'France', 'Germany', 'India', 'China', 'Singapore']
routes = routes[routes['Country'].isin(allowed_countries)]

# Clean up invalid IDs
routes = routes.replace('\\N', pd.NA)
routes = routes.dropna(subset=['Source Airport ID', 'Destination Airport ID'])
routes['Source Airport ID'] = routes['Source Airport ID'].astype(int)
routes['Destination Airport ID'] = routes['Destination Airport ID'].astype(int)

# Create graph from routes
G = nx.Graph()
for _, row in routes.iterrows():
    G.add_edge(row['Source Airport ID'], row['Destination Airport ID'])

import random

# Step 1: Select diverse seed airports
seed_airports = set()
countries_seen = set()
for _, row in airports.iterrows():
    if row['Country'] not in countries_seen and row['Airport ID'] in G:
        seed_airports.add(row['Airport ID'])
        countries_seen.add(row['Country'])
    if len(seed_airports) >= 5:
        break

# Step 3: Add the 20 highest-degree airports
sampled_airports = set(seed_airports)
top20 = sorted(G.nodes, key=lambda n: G.degree[n], reverse=True)[:20]
sampled_airports.update(top20)

# Step 2: Expand seeds by random neighbors
for airport_id in seed_airports:
    neigh = list(G[airport_id])
    if neigh:  # avoid error for isolated airports
        neighbors = random.sample(neigh, min(5, len(neigh)))
        sampled_airports.update(neighbors)

# Step 4: Fill the rest randomly until ~50 total
remaining_airports = list(set(G.nodes) - sampled_airports)
random.shuffle(remaining_airports)
for airport in remaining_airports:
    sampled_airports.add(airport)
    if len(sampled_airports) >= 70:
        break

# Filter routes to sampled airports
filtered_routes = routes[
    routes['Source Airport ID'].isin(sampled_airports) &
    routes['Destination Airport ID'].isin(sampled_airports)
]

# Limit edges per airport
#max_edges_per_airport = 10
#limited_routes_list = []

#airport_edge_count = {airport: 0 for airport in sampled_airports}
#for _, row in filtered_routes.iterrows():
#    source = row['Source Airport ID']
#    dest = row['Destination Airport ID']
#    if airport_edge_count[source] < max_edges_per_airport and airport_edge_count[dest] < max_edges_per_airport:
#        limited_routes_list.append(row)
#        airport_edge_count[source] += 1
#        airport_edge_count[dest] += 1

#limited_routes = pd.DataFrame(limited_routes_list)
limited_routes = filtered_routes

# Ensure the graph is fully connected
G_limited = nx.Graph()
G_limited.add_edges_from([
    (str(row['Source Airport ID']), str(row['Destination Airport ID']))
    for _, row in limited_routes.iterrows()
])

# Keep only the largest connected component
components = list(nx.connected_components(G_limited))
largest_component = max(components, key=len)
graph_nodes = set(largest_component)

# Filter airports and routes again
filtered_airports = airports[airports['Airport ID'].astype(str).isin(graph_nodes)]
limited_routes = limited_routes[
    limited_routes['Source Airport ID'].astype(str).isin(graph_nodes) &
    limited_routes['Destination Airport ID'].astype(str).isin(graph_nodes)
]

# Prepare Graphology-style JSON
graph = {
    "attributes": {},
    "nodes": [],
    "edges": []
}

# Add nodes
for _, row in filtered_airports.iterrows():
    graph["nodes"].append({
        "key": str(row['Airport ID']),
        "attributes": {
            "name": row['Name'],
            "city": row['City'],
            "country": row['Country'],
            "IATA": row['IATA'],
            "ICAO": row['ICAO'],
            "latitude": row['Latitude'],
            "longitude": row['Longitude'],
            "timezone": row['Tz database time zone']
        }
    })

# Create lookup for coordinates
coord_lookup = airports.set_index('Airport ID')[['Latitude', 'Longitude']].to_dict('index')

# Add edges with airline country and geopy distance
for _, row in limited_routes.iterrows():
    src_id = row['Source Airport ID']
    tgt_id = row['Destination Airport ID']
    attr = {
        "airline": row['Airline'],
        "airline_id": row['Airline ID'],
        "airlinecountry": row['Country'],
        "codeshare": True if row['Codeshare'] == 'Y' else False,
        "stops": int(row['Stops']) if pd.notna(row['Stops']) else None,
        "equipment": row['Equipment']
    }

    # Calculate distance using geopy
    if src_id in coord_lookup and tgt_id in coord_lookup:
        coord1 = (coord_lookup[src_id]['Latitude'], coord_lookup[src_id]['Longitude'])
        coord2 = (coord_lookup[tgt_id]['Latitude'], coord_lookup[tgt_id]['Longitude'])
        attr["distance_km"] = round(geopy_distance(coord1, coord2).km, 2)
    else:
        attr["distance_km"] = None

    graph["edges"].append({
        "source": str(src_id),
        "target": str(tgt_id),
        "attributes": attr
    })

# Clean attributes: replace NaNs/NA with None
def clean_attrs(d):
    return {
        k: (None if pd.isna(v) or v is pd.NA or isinstance(v, (pd._libs.missing.NAType, np.float64)) and pd.isna(v) else v)
        for k, v in d.items()
    }

for node in graph["nodes"]:
    node["attributes"] = clean_attrs(node["attributes"])
for edge in graph["edges"]:
    edge["attributes"] = clean_attrs(edge["attributes"])

# Export to file
with open("../data/sampled_data.json", "w") as f:
    json.dump(graph, f, indent=2)

print("Exported sampled_data.json with", len(graph['nodes']), "nodes and", len(graph['edges']), "edges.")

Exported sampled_data.json with 55 nodes and 1542 edges.


In [58]:
import pandas as pd
import json
from geopy.distance import distance as geopy_distance

# -----------------------------
# Load data
# -----------------------------
airports = pd.read_csv("airports.dat", header=None)
routes = pd.read_csv("routes.dat", header=None)
airlines = pd.read_csv("airlines.dat", header=None)

airports.columns = ['Airport ID', 'Name', 'City', 'Country', 'IATA', 'ICAO',
                    'Latitude', 'Longitude', 'Altitude', 'Timezone',
                    'DST', 'Tz database time zone', 'Type', 'Source']
routes.columns = ['Airline', 'Airline ID', 'Source Airport', 'Source Airport ID',
                  'Destination Airport', 'Destination Airport ID', 'Codeshare',
                  'Stops', 'Equipment']
airlines.columns = ['Airline ID', 'Name', 'Alias', 'IATA', 'ICAO',
                    'Callsign', 'Country', 'Active']

# -----------------------------
# Fix data types
# -----------------------------
# Airline ID as string
routes['Airline ID'] = routes['Airline ID'].astype(str)
airlines['Airline ID'] = airlines['Airline ID'].astype(str)

# Remove invalid Airport IDs and convert to int
routes = routes[(routes['Source Airport ID'] != '\\N') & (routes['Destination Airport ID'] != '\\N')]
routes['Source Airport ID'] = routes['Source Airport ID'].astype(int)
routes['Destination Airport ID'] = routes['Destination Airport ID'].astype(int)

# -----------------------------
# Merge airline country into routes
# -----------------------------
routes = routes.merge(
    airlines[['Airline ID', 'Country']],
    on='Airline ID',
    how='left'
)

# Keep only specific countries, others become 'Other'
allowed_countries = ["Canada", "United States", "France", "United Kingdom", "Germany", "China"]
routes['Airline Country'] = routes['Country'].apply(lambda c: c if c in allowed_countries else "Other")

# -----------------------------
# Fixed airport list (46 airports)
# -----------------------------
fixed_iata = [
    # Canada
    "YHD", "YQT", "YTZ", "YSB", "YAM", "YYZ",
    # USA
    "LAX", "IAH", "DTW", "MBP", "DEN", "DFW", "CLT", "ATL", "DRD", "EWR",
    # France
    "CDG", "ORY", "MRS", "LYS", "NCE", "BOD", "TLS", "NTE",
    # UK
    "LGW", "MAN",
    # Germany/Belgium/Switzerland
    "FRA", "MUN", "BSL", "BRU", "DUS",
    # China
    "PEK", "DOY", "NNG", "GYS", "SHA", "CAN", "XIY", "CSX", "SZX", "PVG",
    # West Africa
    "OUA", "ABJ", "DLA", "COD", "SSG"
]

# Map IATA to Airport ID
iata_to_id = airports.set_index('IATA')['Airport ID'].to_dict()
selected_airport_ids = [iata_to_id[iata] for iata in fixed_iata if iata in iata_to_id]

# -----------------------------
# Filter airports & routes
# -----------------------------
filtered_airports = airports[airports['Airport ID'].isin(selected_airport_ids)].copy()

filtered_routes = routes[
    routes['Source Airport ID'].isin(selected_airport_ids) &
    routes['Destination Airport ID'].isin(selected_airport_ids)
].copy()

# Remove isolated airports
connected_ids = set(filtered_routes['Source Airport ID']).union(
    set(filtered_routes['Destination Airport ID'])
)
filtered_airports = filtered_airports[filtered_airports['Airport ID'].isin(connected_ids)].copy()

# Build coordinate lookup
coord_lookup = filtered_airports.set_index('Airport ID')[['Latitude', 'Longitude']].to_dict('index')

# -----------------------------
# Build Graphology-style JSON
# -----------------------------
graph = {"attributes": {}, "nodes": [], "edges": []}

# Add nodes
for _, row in filtered_airports.iterrows():
    graph["nodes"].append({
        "key": str(row['Airport ID']),
        "attributes": {
            "name": row['Name'],
            "city": row['City'],
            "country": row['Country'],
            "IATA": row['IATA'],
            "ICAO": row['ICAO'],
            "latitude": row['Latitude'],
            "longitude": row['Longitude'],
            "timezone": row['Tz database time zone']
        }
    })

# Add edges
for _, row in filtered_routes.iterrows():
    src = row['Source Airport ID']
    tgt = row['Destination Airport ID']
    attr = {
        "airline": row['Airline'],
        "airline_id": row['Airline ID'],
        "airlinecountry": row['Airline Country'],  # updated
        "codeshare": True if row['Codeshare'] == 'Y' else False,
        "stops": int(row['Stops']) if pd.notna(row['Stops']) else None,
        "equipment": row['Equipment']
    }
    if src in coord_lookup and tgt in coord_lookup:
        try:
            attr["distance_km"] = round(
                geopy_distance(
                    (coord_lookup[src]['Latitude'], coord_lookup[src]['Longitude']),
                    (coord_lookup[tgt]['Latitude'], coord_lookup[tgt]['Longitude'])
                ).km, 2
            )
        except:
            attr["distance_km"] = None
    else:
        attr["distance_km"] = None

    graph["edges"].append({"source": str(src), "target": str(tgt), "attributes": attr})

# -----------------------------
# Clean NA / NaN values
# -----------------------------
def clean_attrs(d):
    return {k: (None if pd.isna(v) else v) for k, v in d.items()}

for node in graph["nodes"]:
    node["attributes"] = clean_attrs(node["attributes"])

for edge in graph["edges"]:
    edge["attributes"] = clean_attrs(edge["attributes"])

# -----------------------------
# Export JSON
# -----------------------------
with open("../data/sampled_data.json", "w") as f:
    json.dump(graph, f, indent=2)

print("Exported sampled_data.json with", len(graph['nodes']), "nodes and", len(graph['edges']), "edges.")


Exported sampled_data.json with 43 nodes and 1227 edges.
